In [ ]:
from amuse.community import *
from amuse.community.nbody7.interface import Nbody7
from amuse.ic.plummer import new_plummer_model
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import time
#import subprocess

In [ ]:
NN = 300
converter = nbody_system.nbody_to_si(150*0.7 | units.MSun, 1.0 | units.parsec)
cluster = new_plummer_model(NN, convert_nbody=converter)

In [ ]:
#Check, display files in the current working directory
from pathlib import Path
import shutil
workdir = Path.cwd()
print("Current working directory:", str(workdir))

In [ ]:
archive = workdir / "test"
archive.mkdir(exist_ok=True)
for pat in ("CMPACT", "NBSTAT", "OUT3*", "fort*", "STOP*", "run.out", "err.out"):
    for f in workdir.glob(pat):
        shutil.move(str(f), archive / f.name)
outfile = str(workdir / "run.out")
errfile = str(workdir / "err.out")
print("Output will be written to:", outfile)
print("Errors will be written to:", errfile)
inst = Nbody7(
    converter,
    mode='cpu',
    redirection='file',
    redirect_stdout_file=outfile,
    redirect_stderr_file=errfile,
)
inst.initialize_code()

print("ARCHAIN parameters: CLIGHT NBH IDIS")
print(inst.get_archain_params())
print("Setting new ARCHAIN parameters: ")
inst.set_archain_params(2.5e4, 0, 1)
print("New ARCHAIN parameters:")
print(inst.get_archain_params())

inst.parameters.ETAI = 0.005
inst.parameters.ETAR = 0.005
inst.parameters.RMIN = 1.0e-04
inst.parameters.DTMIN = 1.0e-05
inst.parameters.ETAU = 0.1
inst.parameters.NNBMAX = 32
inst.parameters.DTADJ = 0.5
inst.parameters.DELTAT = 0.5
inst.QE = 1.0e-02
inst.parameters.TCRIT = 1.0e+03
inst.set_kz(27,1)
inst.set_kz(28,3)
inst.commit_parameters()

inst.particles.add_particles(cluster)
inst.particles.name = np.arange(1, NN+1)
inst.particles.kstar = np.zeros(NN, dtype=int)
inst.commit_particles()
print(f"First particle NAME KSTAR (before evolution): {inst.particles[0].name} {inst.particles[0].kstar}")
print(f"Last particle NAME KSTAR (before evolution): {inst.particles[-1].name} {inst.particles[-1].kstar}")


#inst.set_rmin(1.0e-04)
#inst.set_dtmin(1.0e-05)
#print(inst.get_kz(11))
#print(inst.get_rmin())
#print(inst.get_dtmin())
#inst.evolve_model(2.0 * converter.to_si(1.0 | nbody_system.time))
#inst.stop()

In [ ]:
def plot_cluster(fax, converter, inst, boxsize=2.0):
    fax.clear()
    fax.set_xlabel("X (pc)")
    fax.set_ylabel("Y (pc)")
    fax.set_xlim(-boxsize, boxsize)
    fax.set_ylim(-boxsize, boxsize)
    fax.set_aspect('equal')
    ttot = inst.get_time()
    t_unit = converter.to_si(1.0 | nbody_system.time)
    fax.set_title(f"Time = {ttot/t_unit:.2f} Nb time or {ttot.in_(units.Myr)/(1.0 | units.Myr):.2f} Myr")
    for p in inst.particles:
        fax.scatter(p.position[0].in_(units.parsec)/(1.0 | units.parsec), p.position[1].in_(units.parsec)/(1.0 | units.parsec), s=10, color='blue')
    #plt.show()
    #display(fax.figure)

fig, ax = plt.subplots(figsize=(6,6))
plot_cluster(ax, converter, inst, boxsize=3.0)
display(fig)
    

In [ ]:
# The first evolve_model() call lazy-invokes CALL NBODY6 which
# in turn runs SCALE / FPOLY0 / FPOLY2 and populates ZKIN, POT.
# Querying energies BEFORE this would return zeros (the COMMON
# block hasn't been touched yet) so we must establish the
# baseline AFTER an initial step.

t_unit = converter.to_si(1.0 | nbody_system.time)
inst.evolve_model(2.0 * t_unit)
print("1 Nb time =", inst.tstar.in_(units.Myr))
print("1 Nb vel  =", inst.vstar.in_(units.kms))

inst.evolve_model(2.5 * t_unit)
e0 = inst.kinetic_energy + inst.potential_energy
inst.evolve_model(3.0 * t_unit)
e1 = inst.kinetic_energy + inst.potential_energy

ttot = inst.get_time()
print(f"Time after evolution: {ttot/t_unit} Nb time or {ttot.in_(units.Myr)}")

print(f"First particle NAME KSTAR (after evolution): {inst.particles[0].name} {inst.particles[0].kstar}")
print(f"Last particle NAME KSTAR (after evolution): {inst.particles[-1].name} {inst.particles[-1].kstar}")

# Typically, relative enery error is expected to be
# under 1% for short evolutions on a smooth Plummer model.
rel_err = abs((e1 - e0) / e0)
print(f"Relative energy error: {rel_err:.2e}")
assert rel_err < 0.01, "Energy error exceeds 1%"


#inst.cleanup_code()
#inst.stop()

plot_cluster(ax, converter, inst, boxsize=3.0)
display(fig)

In [ ]:
ttot = inst.get_time()
while ttot < 12.0 * t_unit:
    inst.evolve_model(ttot + 1.0 * t_unit)
    ttot = inst.get_time()
    plot_cluster(ax, converter, inst, boxsize=3.0)
    clear_output(wait=True)
    display(fig)
    time.sleep(0.3)

#print(inst.particles[3].position[2].in_(units.parsec)/(1.0 | units.parsec))

In [ ]:
inst.cleanup_code()
inst.stop()